# AIS ETA regression — run on Google Colab

Runs the project pipeline on a Colab VM: download the Tallinn AIS dataset from Kaggle,
load it, and run the **target-options experiment** (`src/target_options.py`), which measures how much
data the observed-arrival target and the crew-ETA target each give.

**Before running**
1. Build the code bundle on your machine: `make colab-bundle` → `dist/ais-eta-regression-src.zip`.
   Put it in the root of your Google Drive (*My Drive*), or upload it when cell 1 asks.
2. Create a Kaggle API token at https://www.kaggle.com/settings/api (*Generate New Token*) and add it to
   Colab **Secrets** (key icon in the left sidebar) as `KAGGLE_API_TOKEN`, with notebook access on.
   A legacy `kaggle.json` works too: add `KAGGLE_USERNAME` and `KAGGLE_KEY` secrets instead.

Runtime: CPU is enough. Download ≈ 3.6 GB; full run ≈ 15 min.

In [ ]:
# 1. Get the code bundle (from Drive if present, else upload)
import os, zipfile, pathlib
from google.colab import drive, files
BUNDLE = "ais-eta-regression-src.zip"
drive.mount("/content/drive")
src = pathlib.Path("/content/drive/MyDrive") / BUNDLE
if not src.exists():
    print(f"{BUNDLE} not found in My Drive; upload it:")
    src = pathlib.Path(next(iter(files.upload())))
zipfile.ZipFile(src).extractall("/content/ais-eta-regression")
os.chdir("/content/ais-eta-regression")
print(sorted(os.listdir()))

In [ ]:
# 2. Install what Colab lacks (Colab's own pandas/numpy/pyarrow are used; versions may differ from requirements.txt)
!pip -q install kaggle==2.2.5 pyyaml
import pandas, numpy, pyarrow, sys
print(sys.version.split()[0], "pandas", pandas.__version__, "numpy", numpy.__version__, "pyarrow", pyarrow.__version__)

In [ ]:
# 3. Kaggle credentials from Colab Secrets (never printed)
from google.colab import userdata
def secret(k):
    try: return userdata.get(k)
    except Exception: return None
tok = secret("KAGGLE_API_TOKEN")
if tok:
    os.environ["KAGGLE_API_TOKEN"] = tok
    p = pathlib.Path.home() / ".kaggle"; p.mkdir(exist_ok=True)
    (p / "access_token").write_text(tok); (p / "access_token").chmod(0o600)
elif secret("KAGGLE_USERNAME") and secret("KAGGLE_KEY"):
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
else:
    raise RuntimeError("Add KAGGLE_API_TOKEN (or KAGGLE_USERNAME + KAGGLE_KEY) in Colab Secrets")
!kaggle datasets files gaborvisky/ais-dataset | head -4

In [ ]:
# 4. Download all 181 days (zip -> one .csv.gz per day)
!python -m src.download --all 2>&1 | tail -3

In [ ]:
# 5. Join position + voyage messages, one parquet per day
!python -m src.load

In [ ]:
# 6. Target-options experiment
!python -m src.target_options > /dev/null
import json
res = json.load(open("reports/target_options.json"))
a, b = res["option_a_observed_arrival"], res["option_b_crew_eta"]
print(f"Days {res['first_day']} .. {res['last_day']} ({res['days']})")
print(f"A observed arrival: {a['arrival_events']:,} arrivals, {a['vessels']} vessels, {a['usable_rows']:,} rows")
print("  hours to arrival:", a["hours_to_arrival"])
print(f"B crew ETA: {b['usable']:,} rows, {b['vessels']} vessels, {b['distinct_destinations']} destinations")
print("  hours to ETA:", b["hours_to_eta"])

In [ ]:
# 7. Save results to Drive (and optionally the processed data, ~1 GB, to skip steps 4-5 next time)
out = pathlib.Path("/content/drive/MyDrive/ais-eta-regression-results"); out.mkdir(exist_ok=True)
!cp reports/target_options.json "{out}/"
SAVE_PROCESSED = False
if SAVE_PROCESSED:
    !cp -r data/processed/loaded "{out}/"
print("Saved to", out)